# Reference Dataset vs Generated — Marginal & Stylized-Facts Analysis

Compares **generated Close log-return paths** (from `generate_samples.py` CSV output)
against a **reference distribution built from the entire `SP500WindowDataset`**
(all sliding windows, L=512, stride=100, over the raw CSV files).

Unlike `OHLC_conditional_csv_analysis.ipynb`, the reference here is **not** limited to
the handful of windows that were sampled — it covers the complete window corpus,
giving a truer population baseline.

Plots produced:
1. Aggregate statistics table (normalized + unnormalized)
2. Marginal distribution (histogram / QQ / ECDF)
3. Thinner marginal distribution (central 96 %)
4. Increments distribution (first differences)
5. Stylized-facts overlap (`sf.distribution`, `sf.acf`, `sf.leverage_effect`)
6. Price sample paths at aggregate level

In [1]:
# ── USER INPUTS ───────────────────────────────────────────────────────────────
CHECKPOINT_STEM = (
    "csv20000_samples5_steps200_seed50_20260607_082851_SDE"
)  # sub-folder under GEN_DIR produced by generate_samples.py

GEN_DIR  = "../../Master-Thesis/data/generated/final/edm"   # base output directory of generate_samples.py
SPLIT    = "both"   # "train" | "val" | "both"

# Reference dataset — the full sliding-window corpus
REFERENCE_ROOT_DIR = "../../Master-Thesis/data/SNP500_individual_normalized_replication"
SEQ_LEN  = 512
STRIDE   = 100
COLUMNS  = ("date", "close", "open", "high", "low")  # must match training config
CLOSE_FEATURE_IDX = 0   # index of Close in the feature dimension (K axis)

# Normalization stats — used only for unnormalized plots
STATS_FILE = "../../Master-Thesis/data/general/normalization_stats_norm_replication.csv"

# Image output
OUTPUT_DIR = "../../Master-Thesis/images/final/edm/reference_vs_generated"
appendix   = "SDE_just_price_paths"  # string appended to output filenames to distinguish checkpoints
# ─────────────────────────────────────────────────────────────────────────────

## 0. Imports

In [2]:
import os

import os
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

import sys
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from scipy import stats as scipy_stats
from statsmodels.tsa.stattools import acf as sm_acf
from pathlib import Path
import warnings

sys.path.insert(0, "..")
sys.path.insert(0, "../../Master-Thesis")

import torch
from torch.utils.data import DataLoader
from src.utils.dataloader import SP500WindowDataset, csdi_collate_fn
import replication.stylized_facts as sf

os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"Output directory: {os.path.abspath(OUTPUT_DIR)}")

Output directory: c:\Users\Lenovo\Documents\SCUOLA\UNI\MASTER\2ANNO\THESIS\Master-Thesis\images\final\edm\reference_vs_generated


## 1. Load generated CSV data

Reads the `*_generated_close.csv` files written by `generate_samples.py`.
Each row is one *(window, sample)* pair; step columns hold the L=512 time steps.
Shapes after parsing:
- `gen_3d`: `(n_gen_windows, n_samples, seq_len)` — generated Close
- `gt_close`: `(n_gen_windows, seq_len)` — ground-truth Close stored alongside (for reference only)

In [3]:
def load_split(split, ckpt_dir):
    gen_path  = os.path.join(ckpt_dir, f"{split}_generated_close.csv")
    ohlc_path = os.path.join(ckpt_dir, f"{split}_gt_ohlc.csv")
    assert os.path.isfile(gen_path),  f"Not found: {gen_path}"
    assert os.path.isfile(ohlc_path), f"Not found: {ohlc_path}"
    df_gen  = pd.read_csv(gen_path)
    df_ohlc = pd.read_csv(ohlc_path)
    step_cols = [c for c in df_gen.columns if c.startswith("step_")]
    print(f"[{split}] windows={df_gen['window_idx'].nunique()}  "
          f"samples={df_gen['sample_idx'].nunique()}  seq_len={len(step_cols)}")
    return df_gen, df_ohlc, step_cols


def parse_arrays(df_gen, df_ohlc, step_cols):
    window_ids = sorted(df_gen["window_idx"].unique())
    n_windows  = len(window_ids)
    n_samples  = df_gen["sample_idx"].nunique()
    seq_len    = len(step_cols)
    gen_3d = np.zeros((n_windows, n_samples, seq_len), dtype=np.float32)
    for wi, wid in enumerate(window_ids):
        sub = df_gen[df_gen["window_idx"] == wid].sort_values("sample_idx")
        gen_3d[wi] = sub[step_cols].to_numpy(dtype=np.float32)
    def gt_feat(feat_name):
        sub = df_ohlc[df_ohlc["feature"] == feat_name].sort_values("window_idx")
        return sub[step_cols].to_numpy(dtype=np.float32)
    return gen_3d, gt_feat("close"), gt_feat("open"), gt_feat("high"), gt_feat("low"), window_ids


ckpt_dir = os.path.join(GEN_DIR, CHECKPOINT_STEM)
assert os.path.isdir(ckpt_dir), f"Directory not found: {ckpt_dir}"

splits_to_run = ["train", "val"] if SPLIT == "both" else [SPLIT]
parsed_gen = {}
for split in splits_to_run:
    try:
        df_gen, df_ohlc, step_cols = load_split(split, ckpt_dir)
        parsed_gen[split] = parse_arrays(df_gen, df_ohlc, step_cols)
    except AssertionError as e:
        print(f"Skipping {split}: {e}")

# Pool all available splits into one flat set for distribution-level comparison
gen_3d_list, gt_close_list = [], []
for g3d, gtc, *_ in parsed_gen.values():
    gen_3d_list.append(g3d)
    gt_close_list.append(gtc)

gen_3d_all  = np.concatenate(gen_3d_list,  axis=0)   # (N_gen, n_samples, L)
gt_close_all = np.concatenate(gt_close_list, axis=0)  # (N_gen, L)

n_gen_windows, n_samples, seq_len = gen_3d_all.shape
steps = np.arange(seq_len)
print(f"\nPooled generated: gen_3d_all={gen_3d_all.shape}  gt_close_all={gt_close_all.shape}")

[train] windows=20000  samples=5  seq_len=512
[val] windows=237  samples=5  seq_len=512

Pooled generated: gen_3d_all=(20237, 5, 512)  gt_close_all=(20237, 512)


## 2. Build the reference window corpus

Instantiates `SP500WindowDataset` with the same `seq_len` and `stride` used during training,
then iterates through **all** windows and collects the Close log-return sequences.

This gives a reference distribution that covers the complete dataset — not just the
handful of windows stored in the generated CSVs.

Shape: `ref_close` → `(n_ref_windows, seq_len)`

In [4]:
print("Building SP500WindowDataset …")
ref_dataset = SP500WindowDataset(
    root_dir=REFERENCE_ROOT_DIR,
    seq_len=SEQ_LEN,
    stride=STRIDE,
    columns=COLUMNS,
    time_mode="global_index",
    cache_data=True,        # load each file once into RAM
    drop_incomplete=True,
)
print(f"  {len(ref_dataset):,} windows across {len(ref_dataset.files)} files")

ref_loader = DataLoader(
    ref_dataset,
    batch_size=512,
    shuffle=False,
    num_workers=0,          # single-threaded for notebook stability
    collate_fn=csdi_collate_fn,
    drop_last=False,
)

close_chunks = []
for batch in ref_loader:
    # observed_data: (B, K, L)  — K features in order matching COLUMNS[1:]
    obs = batch["observed_data"]                          # (B, K, L)
    close_chunks.append(obs[:, CLOSE_FEATURE_IDX, :].numpy())  # (B, L)

ref_close = np.concatenate(close_chunks, axis=0)          # (n_ref_windows, L)
print(f"Reference corpus: ref_close={ref_close.shape}  "
      f"total scalars={ref_close.size:,}")

Building SP500WindowDataset …
  25,226 windows across 210 files
Reference corpus: ref_close=(25226, 512)  total scalars=12,915,712


## 3. Normalization stats and unnormalization helper

Loads the per-feature mean and standard deviation used during dataset normalisation.
The helper `unnorm(arr, feature)` inverts the z-score transform:
$x_{\text{raw}} = x_{\text{norm}} \cdot \sigma + \mu$

In [5]:
stats_df = pd.read_csv(STATS_FILE, header=0, index_col=0)
stats_df.columns = ["mean", "std"]
stats_df.index   = stats_df.index.str.strip().str.lower()
col_stats = stats_df.to_dict(orient="index")

print("Normalization statistics:")
display(stats_df)

def unnorm(arr, feature):
    """Invert z-score normalisation for the given feature."""
    mu  = col_stats[feature]["mean"]
    sig = col_stats[feature]["std"]
    return arr * sig + mu

# Unnormalized arrays (log-returns in original scale)
gen_3d_unnorm  = unnorm(gen_3d_all, "close")    # (N_gen, n_samples, L)
ref_close_unnorm = unnorm(ref_close, "close")    # (n_ref_windows, L)

print(f"\ngen_3d_unnorm  : {gen_3d_unnorm.shape}")
print(f"ref_close_unnorm: {ref_close_unnorm.shape}")

Normalization statistics:


,mean,std
close,0.000451,0.020867
open,0.000265,0.010859
high,0.012179,0.019033
low,-0.011306,0.017881



gen_3d_unnorm  : (20237, 5, 512)
ref_close_unnorm: (25226, 512)


## 4. Flat views for distribution-level comparisons

Collapse window/sample dimensions into 1-D arrays for scalar-level statistics.

| Array | Content | Shape |
|---|---|---|
| `gen_flat` | all generated Close values (normalized) | `(N_gen × n_samples × L,)` |
| `ref_flat` | all reference Close values (normalized) | `(n_ref_windows × L,)` |
| `gen_flat_un` | same, unnormalized | |
| `ref_flat_un` | same, unnormalized | |

In [6]:
gen_flat    = gen_3d_all.ravel()          # normalized
ref_flat    = ref_close.ravel()           # normalized
gen_flat_un = gen_3d_unnorm.ravel()       # unnormalized
ref_flat_un = ref_close_unnorm.ravel()    # unnormalized

print(f"gen_flat : {gen_flat.shape}   ref_flat : {ref_flat.shape}")
print(f"ratio gen/ref scalars : {len(gen_flat)/len(ref_flat):.2f}x")

gen_flat : (51806720,)   ref_flat : (12915712,)
ratio gen/ref scalars : 4.01x


## 5. Aggregate statistics

Summary statistics (mean, std, quantiles, skewness, excess kurtosis) computed
both in **normalized** and **original (unnormalized)** space.

Fat tails should produce excess kurtosis >> 0.
A well-calibrated model should match the reference across all rows.

In [7]:
# Compute increments — shape collapses windows × samples
gen_paths_2d = gen_3d_all.reshape(-1, seq_len)  # (N_gen × n_samples, L)

In [8]:
sf_output_dir = Path(OUTPUT_DIR) / "stylized_facts"
sf_output_dir.mkdir(parents=True, exist_ok=True)

# ── object arrays of 1-D paths (required by sf.acf and sf.leverage_effect) ───
# For the reference we use all windows (one path per window).
# For generated we flatten window × sample so every sample is its own path.
ref_paths_obj = np.empty(len(ref_close), dtype=object)
for i, p in enumerate(ref_close):
    ref_paths_obj[i] = p

gen_paths_obj = np.empty(len(gen_paths_2d), dtype=object)
for i, p in enumerate(gen_paths_2d):
    gen_paths_obj[i] = p

print(f"ref_paths_obj : {ref_paths_obj.shape}  path_len={ref_paths_obj[0].shape}")
print(f"gen_paths_obj : {gen_paths_obj.shape}  path_len={gen_paths_obj[0].shape}")

ref_paths_obj : (25226,)  path_len=(512,)
gen_paths_obj : (101185,)  path_len=(512,)


## 11. Price sample paths — aggregate level

Converts **unnormalized** Close log-returns to price paths with $S_0 = 1$
via cumulative product of $\exp(r_t)$.

A random subset of reference windows and generated paths are overlaid on a single
panel **without any per-window conditioning information** — the goal is a visual
check of the aggregate behaviour (drift, scale of fluctuations, path diversity).

In [9]:
N_PATHS_SHOW = 150    # number of paths shown per group
SEED_SHOW    = 15

rng_show = np.random.default_rng(SEED_SHOW)

# Reference: pick N random windows and convert to price paths
ref_idx   = rng_show.choice(len(ref_close_unnorm), size=min(N_PATHS_SHOW, len(ref_close_unnorm)), replace=False)
ref_price = np.cumprod(np.exp(ref_close_unnorm[ref_idx]), axis=1)  # (N, L)

# Generated: reshape to (N_gen × n_samples, L), pick N random paths
gen_unnorm_2d = gen_3d_unnorm.reshape(-1, seq_len)                 # (N_gen*n_samples, L)
gen_idx       = rng_show.choice(len(gen_unnorm_2d), size=min(N_PATHS_SHOW, len(gen_unnorm_2d)), replace=False)
gen_price     = np.cumprod(np.exp(gen_unnorm_2d[gen_idx]), axis=1) # (N, L)

cmap   = plt.cm.tab20
n_ref  = len(ref_price)
n_gen  = len(gen_price)

fig, axes = plt.subplots(1, 2, figsize=(15, 5), sharey=False)

ax = axes[0]
for i, p in enumerate(ref_price):
    ax.plot(steps, p, color=cmap((i % 20) / 20), lw=0.6, alpha=0.75)
ax.axhline(1.0, color="black", lw=0.7, ls=":", label="$S_0 = 1$")
ax.set_title(f"Reference paths (n={n_ref})")
ax.set_xlabel("step  $t$"); ax.set_ylabel("price  ($S_0 = 1$)")
ax.legend(fontsize=9)

ax = axes[1]
for i, p in enumerate(gen_price):
    ax.plot(steps, p, color=cmap((i % 20) / 20), lw=0.6, alpha=0.75)
ax.axhline(1.0, color="black", lw=0.7, ls=":", label="$S_0 = 1$")
ax.set_title(f"Generated paths (n={n_gen})")
ax.set_xlabel("step  $t$"); ax.set_ylabel("price  ($S_0 = 1$)")
ax.legend(fontsize=9)

plt.suptitle(
    f"Price sample paths — aggregate level  (S₀ = 1, unnormalized log-returns → prices)",
    y=1.01, fontsize=13,
)
plt.tight_layout()
display(fig)
plt.savefig(os.path.join(OUTPUT_DIR, f"price_paths_aggregate_{appendix}.png"), dpi=300, bbox_inches="tight")
plt.close(fig)


<Figure size 1500x500 with 2 Axes>

In [10]:
N_PATHS_SHOW = 500   # number of paths shown per group
SEED_SHOW    = 15
Y_MAX        = 3      # fixed y-axis upper bound; paths exceeding this are excluded

rng_show = np.random.default_rng(SEED_SHOW)

# Reference: pick N random windows and convert to price paths
ref_idx   = rng_show.choice(len(ref_close_unnorm), size=min(N_PATHS_SHOW, len(ref_close_unnorm)), replace=False)
ref_price = np.cumprod(np.exp(ref_close_unnorm[ref_idx]), axis=1)  # (N, L)
ref_price = ref_price[ref_price.max(axis=1) <= Y_MAX]

# Generated: reshape to (N_gen × n_samples, L), pick N random paths
gen_unnorm_2d = gen_3d_unnorm.reshape(-1, seq_len)                 # (N_gen*n_samples, L)
gen_idx       = rng_show.choice(len(gen_unnorm_2d), size=min(N_PATHS_SHOW, len(gen_unnorm_2d)), replace=False)
gen_price     = np.cumprod(np.exp(gen_unnorm_2d[gen_idx]), axis=1) # (N, L)
gen_price     = gen_price[gen_price.max(axis=1) <= Y_MAX]

cmap   = plt.cm.tab20
n_ref  = len(ref_price)
n_gen  = len(gen_price)

fig, axes = plt.subplots(1, 2, figsize=(15, 5), sharey=True)

ax = axes[0]
for i, p in enumerate(ref_price):
    ax.plot(steps, p, color=cmap((i % 20) / 20), lw=0.6, alpha=0.75)
# ax.axhline(1.0, color="black", lw=0.7, ls=":")#, label="$S_0 = 1$")
ax.set_ylim(0, Y_MAX)
# ax.set_title(f"Reference paths (n={n_ref})")
# ax.set_xlabel("step  $t$"); ax.set_ylabel("price  ($S_0 = 1$)")
# ax.legend(fontsize=9)

ax = axes[1]
for i, p in enumerate(gen_price):
    ax.plot(steps, p, color=cmap((i % 20) / 20), lw=0.6, alpha=0.75)
# ax.axhline(1.0, color="black", lw=0.7, ls=":")#, label="$S_0 = 1$")
ax.set_ylim(0, Y_MAX)
# ax.set_title(f"Generated paths (n={n_gen})")
# ax.set_xlabel("step  $t$"); ax.set_ylabel("price  ($S_0 = 1$)")
# ax.legend(fontsize=9)

# plt.suptitle(
#     f"Price sample paths — aggregate level  (S₀ = 1, unnormalized log-returns → prices)",
#     y=1.01, fontsize=13,
# )
plt.tight_layout()
display(fig)
plt.savefig(os.path.join(OUTPUT_DIR, f"price_paths_aggregate_{appendix}.png"), dpi=300, bbox_inches="tight")
plt.close(fig)


<Figure size 1500x500 with 2 Axes>